# Day 4 — Program & high-assurance verification (CBMC · Cryptol · SAW)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/fmaiv/blob/main/notebooks/04_day4_program_verif.ipynb)

Runs the Day-4 verification examples in [`day04/examples/`](https://github.com/ttj/fmaiv/tree/main/day04/examples) — the same files and verdicts CI uses. **CBMC** installs in seconds on Colab; **Cryptol/SAW** are large Galois release tarballs, so they are preinstalled in Codespaces/the course image and skipped on Colab unless you opt in. (The Day-4 *frontier* neural-network examples are separate: `05_day4_nn_robustness.ipynb` and `06_day4_nn_mnist.ipynb`.)

## Setup

In [ ]:
# --- Setup: find the course repo (clone it on Colab), define run helpers ------
# Idempotent: in GitHub Codespaces / the course image the repo and tools are
# already present, so the installs in the next cell are skipped. On Google Colab
# this clones the repo once. Re-running is safe.
import os, sys, re, subprocess, shutil, pathlib

def sh(cmd):
    """Run a shell command, streaming output; raise on failure."""
    print('$', cmd)
    subprocess.run(cmd, shell=True, check=True)

def run(cmd, expect=None):
    """Run a command, show its output, and (optionally) assert a verdict regex
    appears -- so this notebook self-checks exactly like CI (check_examples.sh)."""
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout or '') + (r.stderr or '')
    print(out.rstrip())
    if expect is not None:
        assert re.search(expect, out), f'FAILED: expected /{expect}/ in output'
        print(f'  [ok] matched /{expect}/')
    elif r.returncode != 0:
        raise RuntimeError(f'command exited {r.returncode}')
    return out

def find_repo_root(marker='day01/examples'):
    for d in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]:
        if (d / marker).is_dir():
            return d
    return None

REPO = find_repo_root()
if REPO is None:                       # Colab: no repo on disk -> clone it once
    if not pathlib.Path('fmaiv').exists():
        sh('git clone --depth 1 https://github.com/ttj/fmaiv')
    REPO = pathlib.Path('fmaiv').resolve()
os.chdir(REPO)
assert (REPO / 'day01' / 'examples').is_dir(), 'unexpected repo layout'
print('Course repo:', REPO)

In [ ]:
# CBMC installs from apt in seconds (Colab) / preinstalled (Codespaces). Cryptol
# and SAW are large (~hundreds of MB); preinstalled in Codespaces/the course
# image. To also run the Cryptol/SAW cells on Colab, set this True (slow):
INSTALL_CRYPTOL_SAW = False

if not shutil.which('cbmc'):
    sh('apt-get -qq update && apt-get -qq install -y cbmc')

if INSTALL_CRYPTOL_SAW and not shutil.which('saw'):
    sh('apt-get -qq update && apt-get -qq install -y clang-15 && ln -sf /usr/bin/clang-15 /usr/local/bin/clang')
    sh('curl -L -o /tmp/saw.tgz https://github.com/GaloisInc/saw-script/releases/download/v1.5/'
       'saw-1.5-ubuntu-24.04-X64-with-solvers.tar.gz '
       '&& mkdir -p /opt/saw && tar -xzf /tmp/saw.tgz -C /opt/saw --strip-components=1')
    os.environ['PATH'] = '/opt/saw/bin' + os.pathsep + os.environ['PATH']

for t in ('cbmc', 'cryptol', 'saw', 'clang'):
    print(f'{t:8s}:', shutil.which(t) or 'not found (Codespaces has it; or set INSTALL_CRYPTOL_SAW=True)')

## 1. CBMC — bounded model checking of C
Each check pairs a program with a small harness; `--unwind N` bounds loop unrolling. **`VERIFICATION SUCCESSFUL`** means no assertion, arithmetic-overflow, or array-bounds violation exists within the bound.

In [ ]:
cbmc_checks = [
    ('counter',             'counter.c counter_check.c --unwind 26 --unwinding-assertions'),
    ('array_max',           'array_max.c array_max_check.c --unwind 6 --unwinding-assertions'),
    ('binsearch',           'binsearch.c binsearch_check.c --unwind 10 --unwinding-assertions'),
    ('loop_invariant_demo', 'loop_invariant_demo.c --unwind 21 --unwinding-assertions'),
]
for name, args in cbmc_checks:
    print(f'\n===== cbmc {name} =====')
    run(f'cd day04/examples && cbmc {args}', expect='VERIFICATION SUCCESSFUL')

## 2. Cryptol — bit-precise specs, proved with `:prove`
Cryptol is a DSL for bit-level algorithms; `:prove` discharges each property to an SMT solver and prints **`Q.E.D.`**

In [ ]:
if shutil.which('cryptol'):
    cry = [('counter.cry',    ['bounded_invariant', 'inductive_invariant']),
           ('popcount.cry',   ['popcount_kernighan_eq']),
           ('caesar.cry',     ['roundtrip', 'decrypt_inverts']),
           ('xor_cipher.cry', ['roundtrip', 'involutive'])]
    for f, props in cry:
        for p in props:
            print(f'\n===== cryptol {f} :prove {p} =====')
            run(f'cryptol -c ":prove {p}" day04/examples/{f}', expect='Q.E.D.')
else:
    print('cryptol not on PATH -- preinstalled in Codespaces/the course image.')
    print('On Colab, set INSTALL_CRYPTOL_SAW=True in the setup cell (large download), then re-run.')

## 3. SAW — prove C matches its Cryptol spec
SAW compiles `popcount.c` to LLVM bitcode and proves it is **equivalent** to the Cryptol specification (`Proof succeeded`).

In [ ]:
if shutil.which('saw') and shutil.which('clang'):
    print('===== saw popcount.saw =====')
    run('cd day04/examples && clang -c -emit-llvm -O0 -o popcount.bc popcount.c && saw popcount.saw',
        expect='Proof succeeded')
else:
    print('saw/clang not on PATH -- preinstalled in Codespaces/the course image.')
    print('On Colab, set INSTALL_CRYPTOL_SAW=True in the setup cell, then re-run.')

### Next steps
- **Run it natively, identical tool versions:** `docker compose run --rm day04` (see [`day04/README.md`](https://github.com/ttj/fmaiv/blob/main/day04/README.md)) bundles CBMC + Cryptol + SAW + clang-15.
- **Try it yourself:** the assignment is in [`day04/assignments/day04.md`](https://github.com/ttj/fmaiv/blob/main/day04/assignments/day04.md).
- **Frontier:** `05_day4_nn_robustness.ipynb` (certify a neural network) and `06_day4_nn_mnist.ipynb` (MNIST).
- **Slides:** [Day 4 — Program & high-assurance verification](https://ttj.github.io/fmaiv/day04.html).